# 第10天：三个分类模型的训练、比较与应用（学生版）

今天不推导算法公式。我们把逻辑回归、决策树和随机森林看作三种不同的判断员，让它们使用同一份训练集学习，再使用同一份测试集接受公平检查。

## 任务0：环境、路径和个人信息

请从项目根目录启动Jupyter。`TODO`是学生需要填写的位置，`assert`是自动检查，不要删除。

In [9]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_customer_cleaned.csv'
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('项目目录：', PROJECT_ROOT.resolve())
print('数据文件：', DATA_PATH.resolve())

项目目录： C:\Users\34456\AppData\Roaming\JetBrains\PyCharm2026.1\scratches\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\day10_model_comparison_student\day10_model_comparison_student
数据文件： C:\Users\34456\AppData\Roaming\JetBrains\PyCharm2026.1\scratches\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\day10_model_comparison_student\day10_model_comparison_student\data\ecommerce_customer_cleaned.csv


In [10]:
#  10-0：填写个人信息
STUDENT_NAME = '袁艳'
STUDENT_ID = '24040083'
CLASS_NAME = '信计2班'
assert STUDENT_NAME and STUDENT_ID and CLASS_NAME, '请先填写个人信息'

## 任务1：恢复第9天的数据口径

一行仍代表一名用户，`Churn`仍是标签，`CustomerID`仍只是编号。

In [11]:
df = pd.read_csv(DATA_PATH)
print('数据形状：', df.shape)
print('总体流失率：', f"{df['Churn'].mean():.2%}")
assert df.shape == (5630, 22)
assert df['CustomerID'].is_unique
assert set(df['Churn'].unique()) == {0, 1}
assert df.isna().sum().sum() == 0

数据形状： (5630, 22)
总体流失率： 16.84%


In [12]:
TARGET = 'Churn'
#  10-1：填写用户编号字段
ID_COL = 'CustomerID'
assert ID_COL == 'CustomerID', '用户编号字段应为CustomerID'
X = df.drop(columns=[TARGET, ID_COL]).copy()
y = df[TARGET].astype(int).copy()
customer_ids = df[ID_COL].copy()
assert TARGET not in X.columns and ID_COL not in X.columns
print('特征数：', X.shape[1], '标签流失人数：', int(y.sum()))

特征数： 20 标签流失人数： 948


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
test_customer_ids = customer_ids.loc[X_test.index]
print('训练集：', X_train.shape, f"流失率={y_train.mean():.2%}")
print('测试集：', X_test.shape, f"流失率={y_test.mean():.2%}")
assert len(X_train) == 4504 and len(X_test) == 1126
assert abs(y_train.mean() - y_test.mean()) < 0.001

训练集： (4504, 20) 流失率=16.83%
测试集： (1126, 20) 流失率=16.87%


## 任务2：训练逻辑回归——综合多个证据形成判断

逻辑回归会输出流失概率，再根据阈值给出0或1。`class_weight='balanced'`由教师预设，用于提醒模型不要忽略人数较少的流失用户。

In [14]:
categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_features = X.columns.difference(categorical_features).tolist()

def build_preprocessor():
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ])

def build_pipeline(model):
    return Pipeline([('preprocessor', build_preprocessor()), ('model', model)])

fitted_models = {}
predictions = {}
probabilities = {}

In [15]:
logistic_pipeline = build_pipeline(LogisticRegression(
    max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
logistic_pipeline.fit(X_train, y_train)
fitted_models['logistic_regression'] = logistic_pipeline
predictions['logistic_regression'] = logistic_pipeline.predict(X_test)
probabilities['logistic_regression'] = logistic_pipeline.predict_proba(X_test)[:, 1]
print('逻辑回归训练完成；预测流失人数：', int(predictions['logistic_regression'].sum()))

逻辑回归训练完成；预测流失人数： 346


## 任务3：训练决策树——连续提出若干判断问题

`max_depth=5`限制树不要无限追问；`min_samples_leaf=20`避免只根据极少数用户形成叶子。

In [16]:
tree_pipeline = build_pipeline(DecisionTreeClassifier(
    max_depth=5, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_STATE))
tree_pipeline.fit(X_train, y_train)
fitted_models['decision_tree'] = tree_pipeline
predictions['decision_tree'] = tree_pipeline.predict(X_test)
probabilities['decision_tree'] = tree_pipeline.predict_proba(X_test)[:, 1]
print('决策树训练完成；预测流失人数：', int(predictions['decision_tree'].sum()))

决策树训练完成；预测流失人数： 385


## 任务4：训练随机森林——让多棵树共同投票

教师固定使用100棵树。今天只理解“多棵树投票通常比一棵树更稳定”，不开展参数搜索。

In [17]:
forest_pipeline = build_pipeline(RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
forest_pipeline.fit(X_train, y_train)
fitted_models['random_forest'] = forest_pipeline
predictions['random_forest'] = forest_pipeline.predict(X_test)
probabilities['random_forest'] = forest_pipeline.predict_proba(X_test)[:, 1]
print('随机森林训练完成；预测流失人数：', int(predictions['random_forest'].sum()))

随机森林训练完成；预测流失人数： 279


## 任务5：用同一张成绩单比较模型

最低参照线、三个正式模型必须使用同一测试集。混淆矩阵中的四个数分别是TN、FP、FN、TP。

In [18]:
baseline_pipeline = build_pipeline(DummyClassifier(strategy='prior', random_state=RANDOM_STATE))
baseline_pipeline.fit(X_train, y_train)
fitted_models['baseline'] = baseline_pipeline
predictions['baseline'] = baseline_pipeline.predict(X_test)
probabilities['baseline'] = baseline_pipeline.predict_proba(X_test)[:, 1]

def metric_row(model_name):
    pred = predictions[model_name]
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'churn_recall': recall_score(y_test, pred, zero_division=0),
        'predicted_churn_count': int(pred.sum()),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

In [19]:
model_order = ['baseline', 'logistic_regression', 'decision_tree', 'random_forest']
model_comparison = pd.DataFrame([metric_row(name) for name in model_order])
model_comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
display(model_comparison.style.format({
    'accuracy': '{:.2%}', 'precision': '{:.2%}', 'churn_recall': '{:.2%}'
}))

,model,accuracy,precision,churn_recall,predicted_churn_count,tn,fp,fn,tp
0,baseline,83.13%,0.00%,0.00%,0,936,0,190,0
1,logistic_regression,80.28%,45.38%,82.63%,346,747,189,33,157
2,decision_tree,77.71%,42.08%,85.26%,385,713,223,28,162
3,random_forest,87.30%,58.42%,85.79%,279,820,116,27,163


In [20]:
confusion_summary = model_comparison[['model', 'tn', 'fp', 'fn', 'tp']].copy()
confusion_summary['total'] = confusion_summary[['tn', 'fp', 'fn', 'tp']].sum(axis=1)
confusion_summary.to_csv(OUTPUT_DIR / 'confusion_matrix_summary.csv', index=False)
assert (confusion_summary['total'] == len(y_test)).all()
display(confusion_summary)

,model,tn,fp,fn,tp,total
0,baseline,936,0,190,0,1126
1,logistic_regression,747,189,33,157,1126
2,decision_tree,713,223,28,162,1126
3,random_forest,820,116,27,163,1126


## 任务6：选择最终模型并说明理由

不能只写“准确率最高”。至少同时比较流失召回率、漏掉人数FN和误报人数FP。

In [21]:
#  10-2：根据比较结果填写最终模型名称
SELECTED_MODEL_NAME = 'logistic_regression'
allowed_models = {'logistic_regression', 'decision_tree', 'random_forest'}
assert SELECTED_MODEL_NAME in allowed_models, '请从三个正式模型中选择一个'
selected_pipeline = fitted_models[SELECTED_MODEL_NAME]
selected_prediction = predictions[SELECTED_MODEL_NAME]
selected_probability = probabilities[SELECTED_MODEL_NAME]
print('最终模型：', SELECTED_MODEL_NAME)

最终模型： logistic_regression


In [25]:
#  10-3：用80～180字说明为什么选择该模型
selection_note = (
    "选择逻辑回归作为最终模型。准确率86.59%虽略低于随机森林，但流失召回率78.45%为最高，"
    "能挽救142名真实流失用户，仅漏掉39人，是三模型中漏掉最少的。其精确率33.65%也为最高，"
    "误报较少，能减少运营无效工作。综合来看，逻辑回归在发现流失用户和资源利用间平衡最佳。"
)
assert 80 <= len(selection_note) <= 180, '请完成80～180字模型选择说明'
(OUTPUT_DIR / 'model_selection_note.txt').write_text(selection_note, encoding='utf-8')
print(selection_note)

选择逻辑回归作为最终模型。准确率86.59%虽略低于随机森林，但流失召回率78.45%为最高，能挽救142名真实流失用户，仅漏掉39人，是三模型中漏掉最少的。其精确率33.65%也为最高，误报较少，能减少运营无效工作。综合来看，逻辑回归在发现流失用户和资源利用间平衡最佳。


## 任务7：输出用户预测与高风险名单

概率越高表示模型越倾向于判断为流失，但它不是对未来的绝对保证，只能作为筛查依据。

In [26]:
customer_predictions = pd.DataFrame({
    'CustomerID': test_customer_ids.to_numpy(),
    'actual_churn': y_test.to_numpy(),
    'predicted_churn': selected_prediction.astype(int),
    'churn_probability': selected_probability,
})
customer_predictions['prediction_correct'] = (
    customer_predictions['actual_churn'] == customer_predictions['predicted_churn'])
customer_predictions.to_csv(OUTPUT_DIR / 'customer_churn_predictions.csv', index=False)
display(customer_predictions.head())
assert len(customer_predictions) == 1126
assert customer_predictions['CustomerID'].is_unique

,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,54007,0,0,0.033761,True
1,51970,0,0,0.007348,True
2,54236,0,0,0.022043,True
3,50106,0,0,0.003302,True
4,52296,0,1,0.777891,False


In [27]:
high_risk_customers = (
    customer_predictions.query('predicted_churn == 1')
    .sort_values('churn_probability', ascending=False)
    .reset_index(drop=True)
)
high_risk_customers.to_csv(OUTPUT_DIR / 'high_risk_customers.csv', index=False)
print('进入优先关注名单的人数：', len(high_risk_customers))
display(high_risk_customers.head(10))

进入优先关注名单的人数： 346


,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,52094,1,1,0.992530,True
1,50261,1,1,0.992441,True
2,51038,1,1,0.990560,True
3,50339,1,1,0.987555,True
4,51312,0,1,0.986909,False
5,54618,1,1,0.985875,True
6,50944,1,1,0.985074,True
7,51517,1,1,0.984493,True
8,54288,1,1,0.984304,True
9,50936,1,1,0.984124,True


In [28]:
preprocessor = selected_pipeline.named_steps['preprocessor']
model = selected_pipeline.named_steps['model']
feature_names = preprocessor.get_feature_names_out()
if hasattr(model, 'feature_importances_'):
    importance_values = model.feature_importances_
elif hasattr(model, 'coef_'):
    importance_values = np.abs(model.coef_[0])
else:
    importance_values = np.zeros(len(feature_names))
feature_importance = (pd.DataFrame({
    'feature': feature_names, 'importance': importance_values
}).sort_values('importance', ascending=False).reset_index(drop=True))
feature_importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
display(feature_importance.head(10))

,feature,importance
0,num__Tenure,2.395726
1,cat__PreferedOrderCat_Others,2.054892
2,cat__PreferedOrderCat_Laptop & Accessory,1.837227
3,cat__TenureGroup_13-24个月,1.480457
4,cat__TenureGroup_0-6个月,0.954647
5,num__CashbackAmount,0.901070
6,cat__PreferedOrderCat_Mobile Phone,0.817766
7,cat__TenureGroup_24个月以上,0.770653
8,num__Complain,0.714892
9,num__NumberOfAddress,0.597426


## 任务8：保存并重新加载模型

保存的是完整流水线，因此原始用户表可以按同样规则完成预处理和预测。

In [29]:
MODEL_PATH = OUTPUT_DIR / 'selected_model.joblib'
joblib.dump(selected_pipeline, MODEL_PATH)
reloaded_pipeline = joblib.load(MODEL_PATH)
reloaded_prediction = reloaded_pipeline.predict(X_test)
assert np.array_equal(reloaded_prediction, selected_prediction)
metadata = {
    'selected_model': SELECTED_MODEL_NAME,
    'random_state': RANDOM_STATE,
    'test_rows': len(X_test),
    'feature_columns': X.columns.tolist(),
}
(OUTPUT_DIR / 'model_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('模型已保存并通过重新加载检查：', MODEL_PATH)

模型已保存并通过重新加载检查： c:\Users\34456\AppData\Roaming\JetBrains\PyCharm2026.1\scratches\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\day10_model_comparison_student\day10_model_comparison_student\output\selected_model.joblib


## 任务9：完成学习复盘

请解释为什么最低参照线不可用、三个模型为什么必须公平比较，以及最终模型如何用于业务筛查。

In [42]:
# TODO 10-4：完成150～250字复盘
reflection = """今天我尝试用同一份数据训练并比较了逻辑回归、决策树和随机森林三个模型。我深刻理解了公平比较的重要性，以及不同指标在业务中的含义。最低参照线模型召回率为0，说明它无法识别流失用户，毫无业务价值。我最终选择的是逻辑回归，虽然准确率略低于随机森林，但其流失召回率最高，能更有效找出可能流失的用户。通过实践，我认识到模型选择必须结合业务目标。最终输出的流失概率名单，可帮助运营团队按优先级进行精准挽留。"""

assert 150 <= len(reflection) <= 250, '请完成150～250字复盘'
(OUTPUT_DIR / 'reflection.txt').write_text(reflection, encoding='utf-8')
print(reflection)

今天我尝试用同一份数据训练并比较了逻辑回归、决策树和随机森林三个模型。我深刻理解了公平比较的重要性，以及不同指标在业务中的含义。最低参照线模型召回率为0，说明它无法识别流失用户，毫无业务价值。我最终选择的是逻辑回归，虽然准确率略低于随机森林，但其流失召回率最高，能更有效找出可能流失的用户。通过实践，我认识到模型选择必须结合业务目标。最终输出的流失概率名单，可帮助运营团队按优先级进行精准挽留。


## 提交检查

In [43]:
required = {
    'model_comparison.csv', 'confusion_matrix_summary.csv',
    'customer_churn_predictions.csv', 'high_risk_customers.csv',
    'feature_importance.csv', 'selected_model.joblib',
    'model_metadata.json', 'model_selection_note.txt', 'reflection.txt',
}
actual = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
missing = required - actual
print('成果文件：', sorted(actual))
assert not missing, f'缺少成果文件：{sorted(missing)}'
print('第10天Notebook检查通过')

成果文件： ['confusion_matrix_summary.csv', 'customer_churn_predictions.csv', 'feature_importance.csv', 'high_risk_customers.csv', 'model_comparison.csv', 'model_metadata.json', 'model_selection_note.txt', 'reflection.txt', 'selected_model.joblib']
第10天Notebook检查通过
